# Time-Series Forecasting

This notebook provides time-series forecasting for conversion metrics, enabling proactive planning and resource allocation.

## Purpose
- Decompose time series into trend, seasonality, and noise components
- Forecast future conversion rates and purchase volumes
- Provide confidence intervals for forecasts
- Identify seasonal patterns and trends for business planning

## Data Sources
- Weekly conversion trend data (`data/processed/demo/weekly_conversion.csv`)
- Historical conversion patterns for forecasting

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Load data
weekly_data = pd.read_csv('../data/processed/demo/weekly_conversion.csv')
weekly_data['week_start'] = pd.to_datetime(weekly_data['week_start'])
weekly_data = weekly_data.set_index('week_start').sort_index()

# Exclude partial first week
weekly_data_clean = weekly_data[weekly_data.index > '2020-10-31'].copy()

# Convert conversion rates from percentage to decimal
weekly_data_clean['view_to_checkout_rate'] = weekly_data_clean['view_to_checkout_rate'] / 100
weekly_data_clean['checkout_to_purchase_rate'] = weekly_data_clean['checkout_to_purchase_rate'] / 100
weekly_data_clean['purchase_conversion_rate'] = weekly_data_clean['purchase_conversion_rate'] / 100

print("Weekly data shape:", weekly_data_clean.shape)
print("\nWeekly data sample:")
print(weekly_data_clean.head())
print("\nData summary:")
print(weekly_data_clean.describe())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Purchase conversion rate over time
axes[0].plot(weekly_data_clean.index, weekly_data_clean['purchase_conversion_rate'],
             'o-', linewidth=2, markersize=6, color='#2E86AB')
axes[0].axhline(weekly_data_clean['purchase_conversion_rate'].mean(),
               color='red', linestyle='--', alpha=0.7, label=f'Mean: {weekly_data_clean["purchase_conversion_rate"].mean():.2%}')
axes[0].set_ylabel('Conversion Rate', fontsize=12)
axes[0].set_title('Weekly Purchase Conversion Rate', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: View-to-checkout rate over time
axes[1].plot(weekly_data_clean.index, weekly_data_clean['view_to_checkout_rate'],
             'o-', linewidth=2, markersize=6, color='#A23B72')
axes[1].axhline(weekly_data_clean['view_to_checkout_rate'].mean(),
               color='red', linestyle='--', alpha=0.7, label=f'Mean: {weekly_data_clean["view_to_checkout_rate"].mean():.2%}')
axes[1].set_ylabel('View-to-Checkout Rate', fontsize=12)
axes[1].set_title('Weekly View-to-Checkout Rate', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Checkout-to-purchase rate over time
axes[2].plot(weekly_data_clean.index, weekly_data_clean['checkout_to_purchase_rate'],
             'o-', linewidth=2, markersize=6, color='#F18F01')
axes[2].axhline(weekly_data_clean['checkout_to_purchase_rate'].mean(),
               color='red', linestyle='--', alpha=0.7, label=f'Mean: {weekly_data_clean["checkout_to_purchase_rate"].mean():.2%}')
axes[2].set_ylabel('Checkout-to-Purchase Rate', fontsize=12)
axes[2].set_xlabel('Week', fontsize=12)
axes[2].set_title('Weekly Checkout-to-Purchase Rate', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def decompose_time_series(series, period=4):
    """
    Simple time series decomposition using moving averages.
    
    Parameters:
    - series: pandas Series with datetime index
    - period: Number of periods for seasonal decomposition (4 for monthly seasonality in weekly data)
    
    Returns dict with trend, seasonal, and residual components
    """
    # Calculate trend using centered moving average
    trend = series.rolling(window=period, center=True).mean()
    
    # Calculate detrended series
    detrended = series - trend
    
    # Calculate seasonal component (average seasonal pattern)
    seasonal_avg = detrended.groupby(detrended.index % period).mean()
    seasonal = detrended.index.map(lambda x: seasonal_avg[x % period])
    
    # Calculate residual
    residual = series - trend - seasonal
    
    return {
        'trend': trend,
        'seasonal': seasonal,
        'residual': residual,
        'original': series
    }

# Decompose purchase conversion rate
decomposition = decompose_time_series(weekly_data_clean['purchase_conversion_rate'], period=4)

# Plot decomposition
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

axes[0].plot(decomposition['original'].index, decomposition['original'], 'o-', linewidth=2, markersize=4)
axes[0].set_ylabel('Conversion Rate', fontsize=12)
axes[0].set_title('Original Series', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(decomposition['trend'].index, decomposition['trend'], linewidth=2, color='orange')
axes[1].set_ylabel('Conversion Rate', fontsize=12)
axes[1].set_title('Trend Component', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

axes[2].plot(decomposition['seasonal'].index, decomposition['seasonal'], linewidth=2, color='green')
axes[2].set_ylabel('Conversion Rate', fontsize=12)
axes[2].set_title('Seasonal Component', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

axes[3].plot(decomposition['residual'].index, decomposition['residual'], 'o-', linewidth=1, markersize=4, color='red')
axes[3].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[3].set_ylabel('Conversion Rate', fontsize=12)
axes[3].set_xlabel('Week', fontsize=12)
axes[3].set_title('Residual Component', fontsize=14, fontweight='bold')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print decomposition statistics
print("=== DECOMPOSITION STATISTICS ===")
print(f"\nTrend range: {decomposition['trend'].min():.2%} - {decomposition['trend'].max():.2%}")
print(f"Seasonal range: {decomposition['seasonal'].min():.2%} - {decomposition['seasonal'].max():.2%}")
print(f"Residual std: {decomposition['residual'].std():.2%}")
print(f"\nTrend direction: {'Increasing' if decomposition['trend'].iloc[-1] > decomposition['trend'].iloc[0] else 'Decreasing'}")

## Forecasting Models

Implement multiple forecasting approaches and compare their performance.

In [ ]:
def simple_moving_average_forecast(series, window=4, forecast_periods=4):
    """
    Simple moving average forecast.
    """
    forecasts = []
    for _i in range(forecast_periods):
        forecast = series.tail(window).mean()
        forecasts.append(forecast)
        # Append forecast to series for next iteration
        series = pd.concat([series, pd.Series([forecast])])
    return forecasts

def exponential_smoothing_forecast(series, alpha=0.3, forecast_periods=4):
    """
    Simple exponential smoothing forecast.
    """
    # Fit model on historical data
    smoothed = [series.iloc[0]]
    for i in range(1, len(series)):
        smoothed.append(alpha * series.iloc[i] + (1 - alpha) * smoothed[-1])

    # Generate forecasts
    forecasts = []
    last_smoothed = smoothed[-1]
    for _i in range(forecast_periods):
        forecasts.append(last_smoothed)
        # For simple exponential smoothing, forecast is constant
    return forecasts

def linear_trend_forecast(series, forecast_periods=4):
    """
    Linear trend forecast using ordinary least squares.
    """
    # Create time index
    x = np.arange(len(series))
    y = series.values

    # Fit linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

    # Generate forecasts
    forecasts = []
    for i in range(forecast_periods):
        forecast = intercept + slope * (len(series) + i)
        forecasts.append(forecast)

    return forecasts, {'slope': slope, 'intercept': intercept, 'r_squared': r_value**2}

def calculate_forecast_metrics(actual, forecast):
    """
    Calculate forecast accuracy metrics.
    """
    actual = np.array(actual)
    forecast = np.array(forecast)

    mae = np.mean(np.abs(actual - forecast))
    rmse = np.sqrt(np.mean((actual - forecast)**2))
    mape = np.mean(np.abs((actual - forecast) / actual)) * 100

    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

In [ ]:
# Train-test split (use last 4 weeks for testing)
train_size = len(weekly_data_clean) - 4
train_data = weekly_data_clean['purchase_conversion_rate'].iloc[:train_size]
test_data = weekly_data_clean['purchase_conversion_rate'].iloc[train_size:]

print(f"Training size: {len(train_data)} weeks")
print(f"Test size: {len(test_data)} weeks")

# Generate forecasts using different models
sma_forecast = simple_moving_average_forecast(train_data.copy(), window=4, forecast_periods=4)
es_forecast = exponential_smoothing_forecast(train_data, alpha=0.3, forecast_periods=4)
linear_forecast, linear_params = linear_trend_forecast(train_data, forecast_periods=4)

# Calculate metrics
sma_metrics = calculate_forecast_metrics(test_data, sma_forecast)
es_metrics = calculate_forecast_metrics(test_data, es_forecast)
linear_metrics = calculate_forecast_metrics(test_data, linear_forecast)

print("\n=== FORECAST MODEL COMPARISON ===")
print("\n{:<20} {:>10} {:>10} {:>10}".format("Model", "MAE", "RMSE", "MAPE %"))
print("-" * 55)
print("{:<20} {:>10.4f} {:>10.4f} {:>10.2f}".format("Simple MA", sma_metrics['MAE'], sma_metrics['RMSE'], sma_metrics['MAPE']))
print("{:<20} {:>10.4f} {:>10.4f} {:>10.2f}".format("Exp Smoothing", es_metrics['MAE'], es_metrics['RMSE'], es_metrics['MAPE']))
print("{:<20} {:>10.4f} {:>10.4f} {:>10.2f}".format("Linear Trend", linear_metrics['RMSE'], linear_metrics['RMSE'], linear_metrics['MAPE']))

print("\nLinear Trend Parameters:")
print(f"  Slope: {linear_params['slope']:.6f} (conversion rate change per week)")
print(f"  Intercept: {linear_params['intercept']:.4f}")
print(f"  R-squared: {linear_params['r_squared']:.4f}")

print("\n=== FORECAST MODEL COMPARISON ===")
print("\n{:<20} {:>10} {:>10} {:>10}".format("Model", "MAE", "RMSE", "MAPE %"))
print("-" * 55)
print("{:<20} {:>10.4f} {:>10.4f} {:>10.2f}".format("Simple MA", sma_metrics['MAE'], sma_metrics['RMSE'], sma_metrics['MAPE']))
print("{:<20} {:>10.4f} {:>10.4f} {:>10.2f}".format("Exp Smoothing", es_metrics['MAE'], es_metrics['RMSE'], es_metrics['MAPE']))
print("{:<20} {:>10.4f} {:>10.4f} {:>10.2f}".format("Linear Trend", linear_metrics['RMSE'], linear_metrics['RMSE'], linear_metrics['MAPE']))

print("\nLinear Trend Parameters:")
print(f"  Slope: {linear_params['slope']:.6f} (conversion rate change per week)")
print(f"  Intercept: {linear_params['intercept']:.4f}")
print(f"  R-squared: {linear_params['r_squared']:.4f}")

In [ ]:
# Generate future forecasts using all data
forecast_periods = 8  # Forecast 8 weeks ahead

sma_forecast_future = simple_moving_average_forecast(weekly_data_clean['purchase_conversion_rate'].copy(), window=4, forecast_periods=forecast_periods)
es_forecast_future = exponential_smoothing_forecast(weekly_data_clean['purchase_conversion_rate'], alpha=0.3, forecast_periods=forecast_periods)
linear_forecast_future, _ = linear_trend_forecast(weekly_data_clean['purchase_conversion_rate'], forecast_periods=forecast_periods)

# Create future dates
last_date = weekly_data_clean.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=forecast_periods, freq='W-MON')

# Plot forecasts
fig, ax = plt.subplots(figsize=(14, 7))

# Plot historical data
ax.plot(weekly_data_clean.index, weekly_data_clean['purchase_conversion_rate'],
        'o-', linewidth=2, markersize=6, label='Historical', color='#2E86AB')

# Plot forecasts
ax.plot(future_dates, sma_forecast_future, 's--', linewidth=2, markersize=6, label='Simple MA Forecast', color='#A23B72')
ax.plot(future_dates, es_forecast_future, '^--', linewidth=2, markersize=6, label='Exp Smoothing Forecast', color='#F18F01')
ax.plot(future_dates, linear_forecast_future, 'D--', linewidth=2, markersize=6, label='Linear Trend Forecast', color='#C73E1D')

# Add vertical line at forecast start
ax.axvline(last_date, color='black', linestyle=':', linewidth=2, label='Forecast Start')

ax.set_ylabel('Conversion Rate', fontsize=12)
ax.set_xlabel('Week', fontsize=12)
ax.set_title('Purchase Conversion Rate Forecasts', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Confidence Intervals for Forecasts

Calculate confidence intervals based on historical volatility.

In [ ]:
def calculate_forecast_ci(historical_data, forecast, confidence_level=0.95):
    """
    Calculate confidence intervals for forecasts based on historical volatility.
    """
    # Calculate standard deviation of historical data
    std_dev = historical_data.std()
    
    # Calculate z-score for confidence level
    z_score = stats.norm.ppf((1 + confidence_level) / 2)
    
    # Calculate confidence intervals (widening over time)
    ci_lower = []
    ci_upper = []
    for i, f in enumerate(forecast):
        # Increase uncertainty over time
        uncertainty = std_dev * np.sqrt(1 + i / len(historical_data))
        ci_lower.append(f - z_score * uncertainty)
        ci_upper.append(f + z_score * uncertainty)
    
    return ci_lower, ci_upper

# Calculate confidence intervals for linear trend forecast
ci_lower, ci_upper = calculate_forecast_ci(
    weekly_data_clean['purchase_conversion_rate'],
    linear_forecast_future,
    confidence_level=0.95
)

# Plot with confidence intervals
fig, ax = plt.subplots(figsize=(14, 7))

# Plot historical data
ax.plot(weekly_data_clean.index, weekly_data_clean['purchase_conversion_rate'],
        'o-', linewidth=2, markersize=6, label='Historical', color='#2E86AB')

# Plot forecast
ax.plot(future_dates, linear_forecast_future, 'D--', linewidth=2, markersize=6,
        label='Linear Trend Forecast', color='#C73E1D')

# Plot confidence intervals
ax.fill_between(future_dates, ci_lower, ci_upper, alpha=0.2, color='#C73E1D',
                label='95% Confidence Interval')

# Add vertical line at forecast start
ax.axvline(last_date, color='black', linestyle=':', linewidth=2, label='Forecast Start')

ax.set_ylabel('Conversion Rate', fontsize=12)
ax.set_xlabel('Week', fontsize=12)
ax.set_title('Purchase Conversion Rate Forecast with 95% Confidence Intervals', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print forecast summary
print("=== 8-WEEK FORECAST SUMMARY (Linear Trend) ===")
print("\n{:<12} {:>12} {:>12} {:>12}".format("Week", "Forecast", "Lower 95%", "Upper 95%"))
print("-" * 50)
for _i, (date, forecast, lower, upper) in enumerate(zip(future_dates, linear_forecast_future, ci_lower, ci_upper)):
    print("{:<12} {:>12.2%} {:>12.2%} {:>12.2%}".format(
        date.strftime('%Y-%m-%d'), forecast, lower, upper
    ))

def calculate_forecast_ci(historical_data, forecast, confidence_level=0.95):
    """
    Calculate confidence intervals for forecasts based on historical volatility.
    """
    # Calculate standard deviation of historical data
    std_dev = historical_data.std()

    # Calculate z-score for confidence level
    z_score = stats.norm.ppf((1 + confidence_level) / 2)

    # Calculate confidence intervals (widening over time)
    ci_lower = []
    ci_upper = []
    for i, f in enumerate(forecast):
        # Increase uncertainty over time
        uncertainty = std_dev * np.sqrt(1 + i / len(historical_data))
        ci_lower.append(f - z_score * uncertainty)
        ci_upper.append(f + z_score * uncertainty)

    return ci_lower, ci_upper

# Calculate confidence intervals for linear trend forecast
ci_lower, ci_upper = calculate_forecast_ci(
    weekly_data_clean['purchase_conversion_rate'],
    linear_forecast_future,
    confidence_level=0.95
)

# Plot with confidence intervals
fig, ax = plt.subplots(figsize=(14, 7))

# Plot historical data
ax.plot(weekly_data_clean.index, weekly_data_clean['purchase_conversion_rate'],
        'o-', linewidth=2, markersize=6, label='Historical', color='#2E86AB')

# Plot forecast
ax.plot(future_dates, linear_forecast_future, 'D--', linewidth=2, markersize=6,
        label='Linear Trend Forecast', color='#C73E1D')

# Plot confidence intervals
ax.fill_between(future_dates, ci_lower, ci_upper, alpha=0.2, color='#C73E1D',
                label='95% Confidence Interval')

# Add vertical line at forecast start
ax.axvline(last_date, color='black', linestyle=':', linewidth=2, label='Forecast Start')

ax.set_ylabel('Conversion Rate', fontsize=12)
ax.set_xlabel('Week', fontsize=12)
ax.set_title('Purchase Conversion Rate Forecast with 95% Confidence Intervals', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print forecast summary
print("=== 8-WEEK FORECAST SUMMARY (Linear Trend) ===")
print("\n{:<12} {:>12} {:>12} {:>12}".format("Week", "Forecast", "Lower 95%", "Upper 95%"))
print("-" * 50)
for date, forecast, lower, upper in zip(future_dates, linear_forecast_future, ci_lower, ci_upper):
    print("{:<12} {:>12.2%} {:>12.2%} {:>12.2%}".format(
        date.strftime('%Y-%m-%d'), forecast, lower, upper
    ))

In [ ]:
# Extract month and week information
weekly_data_clean['month'] = weekly_data_clean.index.month
weekly_data_clean['week_of_month'] = (weekly_data_clean.index.day - 1) // 7 + 1

# Calculate monthly averages
monthly_avg = weekly_data_clean.groupby('month')['purchase_conversion_rate'].mean()

# Plot seasonal patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly pattern
month_names = ['Nov', 'Dec', 'Jan']
bars = axes[0].bar(month_names, monthly_avg.values, color=['#2E86AB', '#A23B72', '#F18F01'])
axes[0].set_ylabel('Average Conversion Rate', fontsize=12)
axes[0].set_title('Monthly Seasonal Pattern', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2%}',
                ha='center', va='bottom', fontsize=11)

# Week of month pattern
weekly_avg = weekly_data_clean.groupby('week_of_month')['purchase_conversion_rate'].mean()
axes[1].plot(weekly_avg.index, weekly_avg.values, 'o-', linewidth=2, markersize=8, color='#2E86AB')
axes[1].set_xlabel('Week of Month', fontsize=12)
axes[1].set_ylabel('Average Conversion Rate', fontsize=12)
axes[1].set_title('Week-of-Month Pattern', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(range(1, 5))

plt.tight_layout()
plt.show()

print("=== SEASONAL PATTERN INSIGHTS ===")
print(f"\nHighest month: {month_names[monthly_avg.argmax()]} ({monthly_avg.max():.2%})")
print(f"Lowest month: {month_names[monthly_avg.argmin()]} ({monthly_avg.min():.2%})")
print(f"Month-to-month variation: {(monthly_avg.max() - monthly_avg.min()):.2%}")

print(f"\nHighest week of month: Week {weekly_avg.argmax()} ({weekly_avg.max():.2%})")
print(f"Lowest week of month: Week {weekly_avg.argmin()} ({weekly_avg.min():.2%})")

## Purchase Volume Forecasting

Forecast actual purchase volumes (not just rates) for inventory planning.

In [ ]:
# Forecast purchase volumes
purchase_volume_forecast, volume_params = linear_trend_forecast(
    weekly_data_clean['purchase_sessions'],
    forecast_periods=8
)

# Calculate confidence intervals
vol_ci_lower, vol_ci_upper = calculate_forecast_ci(
    weekly_data_clean['purchase_sessions'],
    purchase_volume_forecast,
    confidence_level=0.95
)

# Plot purchase volume forecast
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(weekly_data_clean.index, weekly_data_clean['purchase_sessions'],
        'o-', linewidth=2, markersize=6, label='Historical Purchases', color='#2E86AB')
ax.plot(future_dates, purchase_volume_forecast, 'D--', linewidth=2, markersize=6,
        label='Forecast', color='#C73E1D')
ax.fill_between(future_dates, vol_ci_lower, vol_ci_upper, alpha=0.2, color='#C73E1D',
                label='95% Confidence Interval')
ax.axvline(last_date, color='black', linestyle=':', linewidth=2, label='Forecast Start')

ax.set_ylabel('Weekly Purchases', fontsize=12)
ax.set_xlabel('Week', fontsize=12)
ax.set_title('Weekly Purchase Volume Forecast', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=== PURCHASE VOLUME FORECAST (8 weeks) ===")
print(f"\nTrend: {'Increasing' if volume_params['slope'] > 0 else 'Decreasing'} ({volume_params['slope']:.2f} purchases/week)")
print(f"R-squared: {volume_params['r_squared']:.4f}")
print("\n{:<12} {:>12} {:>12} {:>12}".format("Week", "Forecast", "Lower 95%", "Upper 95%"))
print("-" * 50)
for _i, (date, forecast, lower, upper) in enumerate(zip(future_dates, purchase_volume_forecast, vol_ci_lower, vol_ci_upper)):
    print("{:<12} {:>12.0f} {:>12.0f} {:>12.0f}".format(
        date.strftime('%Y-%m-%d'), forecast, lower, upper
    ))

In [ ]:
# Calculate business metrics
avg_weekly_purchases = weekly_data_clean['purchase_sessions'].mean()
forecast_weekly_purchases = np.mean(purchase_volume_forecast)
forecast_change_pct = ((forecast_weekly_purchases - avg_weekly_purchases) / avg_weekly_purchases) * 100

# Assuming AOV of $75
aov = 75
current_weekly_revenue = avg_weekly_purchases * aov
forecast_weekly_revenue = forecast_weekly_purchases * aov
revenue_change = forecast_weekly_revenue - current_weekly_revenue

print("=== BUSINESS IMPACT SUMMARY ===")
print(f"\nCurrent average weekly purchases: {avg_weekly_purchases:.0f}")
print(f"Forecast average weekly purchases: {forecast_weekly_purchases:.0f}")
print(f"Projected change: {forecast_change_pct:+.1f}%")

print(f"\nCurrent weekly revenue (${aov} AOV): ${current_weekly_revenue:,.0f}")
print(f"Forecast weekly revenue (${aov} AOV): ${forecast_weekly_revenue:,.0f}")
print(f"Revenue change: ${revenue_change:+,.0f}/week")
print(f"Annualized revenue change: ${revenue_change * 52:+,.0f}")

print("\n--- FORECAST UNCERTAINTY ---")
forecast_volatility = np.std(purchase_volume_forecast)
print(f"Forecast volatility (std): {forecast_volatility:.0f} purchases/week")
print(f"Revenue volatility: ${forecast_volatility * aov:,.0f}/week")

print("\n--- RECOMMENDATIONS ---")
if forecast_change_pct > 5:
    print("• Inventory: Plan for increased inventory based on upward trend")
    print("• Staffing: Consider staffing adjustments for higher demand")
elif forecast_change_pct < -5:
    print("• Inventory: Reduce inventory orders to match declining trend")
    print("• Marketing: Investigate causes of decline and plan recovery campaigns")
else:
    print("• Inventory: Maintain current inventory levels")
    print("• Monitoring: Continue close monitoring of conversion trends")

print("• Risk Management: Use confidence intervals for contingency planning")
print("• Review: Re-forecast monthly with updated data")

# Calculate business metrics
avg_weekly_purchases = weekly_data_clean['purchase_sessions'].mean()
forecast_weekly_purchases = np.mean(purchase_volume_forecast)
forecast_change_pct = ((forecast_weekly_purchases - avg_weekly_purchases) / avg_weekly_purchases) * 100

# Assuming AOV of $75
aov = 75
current_weekly_revenue = avg_weekly_purchases * aov
forecast_weekly_revenue = forecast_weekly_purchases * aov
revenue_change = forecast_weekly_revenue - current_weekly_revenue

print("=== BUSINESS IMPACT SUMMARY ===")
print(f"\nCurrent average weekly purchases: {avg_weekly_purchases:.0f}")
print(f"Forecast average weekly purchases: {forecast_weekly_purchases:.0f}")
print(f"Projected change: {forecast_change_pct:+.1f}%")

print(f"\nCurrent weekly revenue (${aov} AOV): ${current_weekly_revenue:,.0f}")
print(f"Forecast weekly revenue (${aov} AOV): ${forecast_weekly_revenue:,.0f}")
print(f"Revenue change: ${revenue_change:+,.0f}/week")
print(f"Annualized revenue change: ${revenue_change * 52:+,.0f}")

print("\n--- FORECAST UNCERTAINTY ---")
forecast_volatility = np.std(purchase_volume_forecast)
print(f"Forecast volatility (std): {forecast_volatility:.0f} purchases/week")
print(f"Revenue volatility: ${forecast_volatility * aov:,.0f}/week")

print("\n--- RECOMMENDATIONS ---")
if forecast_change_pct > 5:
    print("• Inventory: Plan for increased inventory based on upward trend")
    print("• Staffing: Consider staffing adjustments for higher demand")
elif forecast_change_pct < -5:
    print("• Inventory: Reduce inventory orders to match declining trend")
    print("• Marketing: Investigate causes of decline and plan recovery campaigns")
else:
    print("• Inventory: Maintain current inventory levels")
    print("• Monitoring: Continue close monitoring of conversion trends")

print("• Risk Management: Use confidence intervals for contingency planning")
print("• Review: Re-forecast monthly with updated data")

In [ ]:
# Create forecast dataframe
forecast_df = pd.DataFrame({
    'week_start': future_dates,
    'conversion_rate_forecast': linear_forecast_future,
    'conversion_rate_lower_95': ci_lower,
    'conversion_rate_upper_95': ci_upper,
    'purchase_volume_forecast': purchase_volume_forecast,
    'purchase_volume_lower_95': vol_ci_lower,
    'purchase_volume_upper_95': vol_ci_upper
})

# Save to data/processed/demo/
output_path = Path('../data/processed/demo/forecast_results.csv')
forecast_df.to_csv(output_path, index=False)
print(f"Forecast results saved to {output_path}")

# Save model parameters
model_params = {
    'linear_trend_slope': volume_params['slope'],
    'linear_trend_intercept': volume_params['intercept'],
    'linear_trend_r_squared': volume_params['r_squared'],
    'historical_std_conversion': weekly_data_clean['purchase_conversion_rate'].std(),
    'historical_std_purchases': weekly_data_clean['purchase_sessions'].std(),
    'forecast_horizon_weeks': forecast_periods
}

import json
params_path = Path('../data/processed/demo/forecast_model_params.json')
with open(params_path, 'w') as f:
    json.dump(model_params, f, indent=2)
print(f"Model parameters saved to {params_path}")

print("\nForecast data preview:")
print(forecast_df.head())

## Summary and Key Takeaways

### Key Findings:
1. **Trend Analysis**: Conversion rates show a [increasing/decreasing/stable] trend over the analysis period
2. **Seasonality**: Strong seasonal patterns observed with peak in [month]
3. **Forecast Accuracy**: Linear trend model provides reasonable forecasts with R-squared of X
4. **Business Impact**: Forecast suggests [X%] change in weekly purchases

### Business Recommendations:
1. **Inventory Planning**: Use purchase volume forecasts for inventory optimization
2. **Resource Allocation**: Align staffing and resources with forecasted demand
3. **Marketing Strategy**: Plan campaigns around seasonal peaks
4. **Risk Management**: Use confidence intervals for contingency planning
5. **Continuous Monitoring**: Re-forecast monthly with updated data

### Limitations:
1. Limited historical data (only 3 months)
2. Holiday season may not represent typical patterns
3. External factors (e.g., promotions, events) not captured
4. Forecasts assume historical patterns continue

## Summary and Key Takeaways

### Calculation Methodology Verification

**Time-Series Forecasting Methods:**
- **Linear Trend**: Ordinary Least Squares (OLS) regression - appropriate for detecting trends
- **Simple Moving Average**: Window-based averaging - good for smoothing short-term noise
- **Exponential Smoothing**: Weighted averaging with decay factor - suitable for stable series

**Confidence Interval Calculation:**
- Based on historical standard deviation
- Uses z-score for normal distribution (appropriate for larger samples)
- Uncertainty increases over time (standard practice in forecasting)

**Data Format Note:**
- Conversion rates in CSV are stored as percentages (e.g., 17.32 = 17.32%)
- All calculations convert to decimals (divide by 100) for mathematical operations
- Final displays convert back to percentages for readability

**Assumptions & Limitations:**
1. **Limited Historical Data**: Only 3 months of data (Nov 2020 - Jan 2021), which includes holiday season patterns that may not represent typical behavior
2. **Small Sample Size**: 14 weeks is insufficient for robust seasonal pattern detection
3. **Holiday Bias**: December peak may be holiday-driven rather than representative
4. **External Factors**: Forecasts don't account for promotions, events, economic changes, or competitive actions
5. **No Causal Modeling**: These are statistical extrapolations, not causal models
6. **Stationarity Assumption**: Linear trend assumes historical patterns continue (may not hold)

**Industry Best Practices Applied:**
- Multiple model comparison (select best fit)
- Train-test split for validation
- Confidence intervals for uncertainty quantification
- Seasonal decomposition (trend, seasonality, noise)
- Business impact translation (purchases to revenue)

### Key Findings:
1. **Trend Analysis**: Conversion rates show a [increasing/decreasing/stable] trend over the analysis period
2. **Seasonality**: Strong seasonal patterns observed with peak in [month]
3. **Forecast Accuracy**: Linear trend model provides reasonable forecasts with R-squared of X
4. **Business Impact**: Forecast suggests [X%] change in weekly purchases

### Business Recommendations:
1. **Inventory Planning**: Use purchase volume forecasts for inventory optimization
2. **Resource Allocation**: Align staffing and resources with forecasted demand
3. **Marketing Strategy**: Plan campaigns around seasonal peaks
4. **Risk Management**: Use confidence intervals for contingency planning
5. **Continuous Monitoring**: Re-forecast monthly with updated data

### Limitations:
1. Limited historical data (only 3 months)
2. Holiday season may not represent typical patterns
3. External factors (e.g., promotions, events) not captured
4. Forecasts assume historical patterns continue